# Natural-Distribution XGBoost Experiment

This notebook trains a multiclass XGBoost classifier on a stratified sample that preserves the dataset's natural class distribution.

**Purpose:** establish a strong overall-accuracy baseline and observe how class imbalance affects minority attack classes.

> Use only authorized data. The dataset and generated model artifacts are not included in this repository.


In [ ]:
import os
from pathlib import Path

import boto3
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

DATA_S3_URI = os.getenv("SECURITY_DATA_S3_URI", "s3://YOUR-BUCKET/analytics/")
MODEL_OUTPUT_DIR = Path(os.getenv("MODEL_OUTPUT_DIR", "../models"))
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = [
    "Dst Port",
    "Protocol",
    "Flow Duration",
    "Flow Byts/s",
    "Flow Pkts/s",
    "Tot Fwd Pkts",
    "Tot Bwd Pkts",
]


## Load the network telemetry

Set `SECURITY_DATA_S3_URI` before running. The source can be an S3 Parquet prefix or another path supported by pandas/pyarrow.


In [ ]:
df = pd.read_parquet(DATA_S3_URI)

print("Rows and columns:", df.shape)
print(df["Label"].value_counts())


## Create a reproducible stratified sample

A one-million-row sample reduces memory requirements while preserving the original label proportions.


In [ ]:
sample_size = min(1_000_000, len(df))

df_sample, _ = train_test_split(
    df,
    train_size=sample_size,
    stratify=df["Label"],
    random_state=42,
)

X = df_sample[FEATURES]
encoder = LabelEncoder()
y = encoder.fit_transform(df_sample["Label"])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Classes:", list(encoder.classes_))


## Train the model

The parameters match the original class experiment and provide a reproducible baseline rather than a production-tuned model.


In [ ]:
natural_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softmax",
    num_class=len(encoder.classes_),
    random_state=42,
    n_jobs=-1,
)

natural_model.fit(X_train, y_train)


## Evaluate performance

Review macro-averaged scores in addition to accuracy. A large gap between weighted and macro scores indicates uneven performance across classes.


In [ ]:
predictions = natural_model.predict(X_test)

print(classification_report(
    y_test,
    predictions,
    target_names=encoder.classes_,
    zero_division=0,
))

print("Training accuracy:", natural_model.score(X_train, y_train))
print("Test accuracy:", natural_model.score(X_test, y_test))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=encoder.classes_,
    xticks_rotation=90,
)
plt.tight_layout()
plt.show()


## Inspect feature importance

Feature importance is descriptive, not causal. It should not be interpreted as proof that one field alone identifies malicious traffic.


In [ ]:
importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": natural_model.feature_importances_,
}).sort_values("importance", ascending=False)

importance


## Save trusted local artifacts

The dashboard expects the model and label encoder from the same training run. Never load pickle or joblib files from an untrusted source.


In [ ]:
joblib.dump(natural_model, MODEL_OUTPUT_DIR / "xgboost_natural.pkl")
joblib.dump(encoder, MODEL_OUTPUT_DIR / "label_encoder.pkl")

print("Saved artifacts to", MODEL_OUTPUT_DIR.resolve())


## Optional S3 upload

Keep model storage private and use a dedicated IAM role. Uncomment only after setting `MODEL_BUCKET` to a bucket you control.


In [ ]:
# MODEL_BUCKET = os.getenv("MODEL_BUCKET")
# if MODEL_BUCKET:
#     s3 = boto3.client("s3")
#     s3.upload_file(
#         MODEL_OUTPUT_DIR / "xgboost_natural.pkl",
#         MODEL_BUCKET,
#         "models/xgboost_natural.pkl",
#     )
#     s3.upload_file(
#         MODEL_OUTPUT_DIR / "label_encoder.pkl",
#         MODEL_BUCKET,
#         "models/label_encoder.pkl",
#     )


## Interpretation

The natural-distribution experiment achieved high overall accuracy in the original test, but rare attack classes were much less reliable. Treat it as a baseline and compare it with the balanced-sampling notebook before making any operational conclusions.
